# Structure of the code
Start after expanding in St:
$$
\dot {\mathbf x }= -St \frac{b'}{\tanh(h)}e_z + \epsilon \left(\mathbf u+St(b-1)\frac{d\mathbf u}{dt}+St^2(1-b)\frac{d^2\mathbf u}{dt^2}\right)
$$
- Expand Lagrangian derivatives of $\mathbf u$ in $\epsilon$ and $St$
- Calculate $\mathbf x$ up to first order explicitly
- Expand those in St again (because they will have some 1/St factor in them)
- Evaluate all terms from the first equation at the first order path, Taylor-expand the trigonometric and hyperbolic functions and drop higher order terms

Note that 
$$
    \frac{\mathrm d u}{\mathrm dt} = \partial_t u + (\dot x,\dot z)\nabla u.
$$
Also, we have 
$$
    \dot x = \epsilon \left(u - \mathrm{St} b'  \frac{\partial d u}{\partial dt}\right) + \mathcal O(\epsilon^2)+\mathcal O(\epsilon \mathrm{St}) + \mathcal O(\mathrm{St}^2),
$$
etc.

In [1]:

ClearAll["Global`*"]
alpha := 1/Sinh[h] * Cosh[z+h];
gamma := 1/Sinh[h] * Sinh[z+h];
beta := Cos[x-t];
delta := Sin[x-t];
u := alpha * beta;
w := gamma * delta;

In [9]:
(*need a proper simplify and expand function*)

Collect[Expand[eps*St+St*eps],{St,eps},Simplify];

op[var_List, max_][p__] :=
  #*Boole[max >= Tr @ Exponent[#, var]] & /@ {p} // Total

xdot2ndorder1 :=eps * (u - St *bp* D[u,t]);
zdot2ndorder1 :=St*(-bp/theta) + eps*(w + St*(-bp) * D[w,t]);
matD[a_] := D[a,t]+{xdot2ndorder1,zdot2ndorder1}.{D[a,x],D[a,z]};

In [15]:
(*Collect first order terms in order to integrate*)
ClearAll[theta];
ClearAll[wfrmxdotfirstorder,wfrmzdotfirstorder,wfrmxfirstorder,wfrmzfirstorder];
wfrmxdotfirstorder = Collect[op[{St,eps},3]@@Expand[eps*(u - St*bp*matD[u]+St^2*bp*matD[matD[u]])],{St,eps},Simplify];
wfrmzdotfirstorder = Collect[op[{St,eps},3]@@Expand[-St*bp/theta + eps*(w - St*bp*matD[w] + St^2*bp*matD[matD[w]])],{St,eps},Simplify];

evalzo = {x -> x0, z -> z0-St*bp*t/theta};


In [21]:
(*Print["\\dot{x}  ="]
Print[wfrmxdotfirstorder//TeXForm]
Print["\\dot{z}  ="]
Print[wfrmzdotfirstorder//TeXForm]*)

We need to hard-code the expansion of the denominator that appears post integrating:

\begin{equation*}
    \frac{1}{(b'\mathrm{St}-i\theta)(b'\mathrm{St}+i\theta)} = 
    \frac{1}{\theta^2}\cdot\frac{1}{(b'\mathrm{St} / \theta - i )(b'\mathrm{St} / \theta + i)} = \frac{1}{\theta}\cdot\frac{1}{1-[-(b'\mathrm{St} / \theta)^2]}
    = \frac{1}{\theta^2}\sum_{k\ge 0}[-(b'\mathrm{St} / \theta)^2]^k
    = \frac{1}{\theta^2}[1 -(b'\mathrm{St} / \theta)^2] + \mathcal{O}(\mathrm{St}^4)
\end{equation*}

In [22]:
expandoneoverA = {(bp St - I theta)^(-1)(bp St + I theta)^(-1) -> (1-bp^2 St^2/theta^2)/theta^2,
        (bp^2 St^2 + theta^2)^(-1) -> ((1-bp^2 St^2 / theta^2)/theta^2),
        (bp^2  St^2  theta + theta^3)^(-1) -> ((1-bp^2 St^2 / theta^2)/theta^3)};
(*Integrate and expand*)
wfrmxfirstordere         = Collect[-dv+op[{St,eps},3]@@(Expand[dv+Integrate[wfrmxdotfirstorder /. evalzo             ,t]  /. expandoneoverA]),{St,eps},Simplify];
wfrmzfirstordere         = Collect[-dv+op[{St,eps},3]@@(dv+Expand[Integrate[wfrmzdotfirstorder /. evalzo             ,t]  /. expandoneoverA]),{St,eps},Simplify];

In [26]:
(*Print[wfrmxfirstordere]*)

We have now found $x, z$ up to $\mathcal O(\epsilon)$ and up to third order in $\epsilon, \mathrm{St}$. Now we simply need to evaluate the original expression:

\begin{equation*}
    \dot{\vec{x} } = \epsilon \vec{v}  = -\mathrm{St} \frac{b'}{\tanh(h)} \vec{e} _{z} + \epsilon
    \left( \vec{u}  + \mathrm{St}(b-1) \frac{\,\mathrm{d}\vec{u} }{\,\mathrm{d}t} +
    \mathrm{St}^2(1-b) \frac{\,\mathrm{d}^2 \vec{u} }{\,\mathrm{d}t^2} \right)
\end{equation*} 
at this first order path and Taylor-expand the trigonometric and hyperbolic functions. This is an example of what happens for $\alpha\beta$:
\begin{equation*}
\alpha\beta(\mathbf x_0+\epsilon \mathbf x_1)
= \alpha\beta(\mathbf x_0) + \epsilon \mathbf x_1\nabla(\alpha\beta)|_{\mathbf x_0}.
\end{equation*}

In [27]:
(*just plugging in*)

xdot1st = eps*(u-St*bp*matD[u]+St^2*bp*matD[matD[u]]) ;
zdot1st = -bp*St/theta + eps*(w - St*bp*matD[w] + St^2*bp*matD[matD[w]]);
(*Expand*)
xdotexp = (xdot1st + {wfrmxfirstordere,wfrmzfirstordere + St*bp*t/theta}.{D[xdot1st,x],D[xdot1st,z]}) /. evalzo;
zdotexp = (zdot1st + {wfrmxfirstordere,wfrmzfirstordere + St*bp*t/theta}.{D[zdot1st,x],D[zdot1st,z]}) /. evalzo;

In [33]:
(*Average over one wave period*)
firstavgrules = {
  Sin[2 (t - x0) | 2 t - 2 x0 | 2 (-x0 + t) | t - x0]^2 -> 1/2,
  Cos[2 (t - x0) | 2 t - 2 x0 | 2 (-x0 + t) | t - x0]^2 -> 1/2
};
(*???NEED to include rules for averaging higher (general?) powers of the sin and cos*)
secondavgrules = {
  Sin[2 (t - x0) | 2 t - 2 x0 | 2 (-x0 + t) | t - x0] -> 0,
  Cos[2 (t - x0) | 2 t - 2 x0 | 2 (-x0 + t) | t - x0] -> 0
};

xdotexp2 = Expand[xdotexp] /. firstavgrules /. secondavgrules;
zdotexp2 = Expand[zdotexp] /. firstavgrules /. secondavgrules;


In [39]:
xdotavgdfinished = Collect[op[{St,eps},4]@@Expand[xdotexp2],{St,eps},Simplify];
zdotavgdfinished = Collect[op[{St,eps},4]@@Expand[zdotexp2],{St,eps},Simplify];
ClearAll[h];
Print["Finished x dot"]
Print[xdotavgdfinished /. theta -> Tanh[h]]
Print["Finished z dot"]
Print[zdotavgdfinished /. theta -> Tanh[h]]

Finished x dot
   2                                           2
eps  Cosh[2 (h + z0 - bp St t Coth[h])] Csch[h]
------------------------------------------------ - 
                       2
 
           2   2        2        2
>   (bp eps  St  Coth[h]  Csch[h]  
 
>      (-2 bp Sinh[2 (h + z0 - bp St t Coth[h])] Tanh[h] + 
 
                                                         2
>        Cosh[2 (h + z0 - bp St t Coth[h])] (bp + Tanh[h] ))) / 2
Finished z dot
                           2                2
St (-(bp Coth[h]) - (bp eps  Coth[h] Csch[h]  
 
>        (Cosh[2 (h + z0 - bp St t Coth[h])] + 
 
>          Sinh[2 (h + z0 - bp St t Coth[h])] Tanh[h])) / 2)


# Sanity checks
We compare our asymptotics to known results in the $\theta = 1$, the $\mathrm{St}\to 0$ and the $h \to \infty$ cases.

In [97]:
(*Sanity checks*)
Print["x dot at St = 0"]
Print[xdotavgdfinished /. theta -> Tanh[h] /. {St -> 0}]
Print["z dot at St = 0"]
Print[zdotavgdfinished /. theta -> Tanh[h] /. {St -> 0}]
Print["x dot at eps = 0"]
Print[xdotavgdfinished /. theta -> Tanh[h] /. {eps -> 0}]
Print["z dot at eps = 0"]
Print[zdotavgdfinished /. theta -> Tanh[h] /. {eps -> 0}]

x dot at St = 0
   2                         2
eps  Cosh[2 (h + z0)] Csch[h]
------------------------------
              2
z dot at St = 0
0
x dot at eps = 0
0
z dot at eps = 0
-(bp St Coth[h])


In [189]:
(*Some previous results to compare against*)
xdotavgd = eps^2/2*((1-2*St^2*bp+St^2*(1-bp)*bp)*(alpha^2+gamma^2)+4*St^2*bp^2*alpha*gamma);
zdotavgd = -St*bp+eps^2/2*(-St*bp*(alpha^2+gamma^2)-St*bp*2*alpha*gamma);

xdotavgdinfdepth = eps^2*E^(2(z0 - bp*St*t))*(1-St^2*bp*(1-bp));
zdotavgdinfdepth = -St*bp - 2*E^(2*(z0 - bp*St*t))*eps^2*St*bp;

In [111]:
(*Verifying the result*)
(*Check that if theta is set to 1 at the end, we get the same result as setting theta = 1 in the beginning*)
evaltheta1 = {theta -> 1};
Print["Difference for theta=1 in x"]
Print[Collect[Expand[(xdotavgdfinished /. evaltheta1) - (xdotavgd /. evalzo /. evaltheta1)], {St, eps}, Simplify]];
Print["Difference for theta=1 in z"]
Print[Collect[Expand[(zdotavgdfinished /. evaltheta1) - (zdotavgd /. evalzo /. evaltheta1)], {St, eps}, Simplify]];
(*Check that in the limit of h → ∞ (theta → 1) we recover the infinite depth asymptotics*)
Print["Difference in the h → ∞-limit in x"]
Print[Collect[Expand[Limit[(xdotavgdfinished /. evaltheta1) - (xdotavgdinfdepth /. evalzo /. evaltheta1), h -> Infinity]], {St, eps}, Simplify]];
Print["Difference in the h → ∞-limit in z"]
Print[Collect[Expand[Limit[(zdotavgdfinished /. evaltheta1) - (zdotavgdinfdepth /. evalzo /. evaltheta1), h -> Infinity]], {St, eps}, Simplify]];

Difference for theta=1 in x
0
Difference for theta=1 in z
0
Difference in the h -> Infinity-limit in x
0
Difference in the h -> Infinity-limit in z
0


In [250]:
(*Comparing with the new simplifications we came up with*)
zSt = h+z0-St*bp*t/Tanh[h];
(*xdotsimplified = eps^2/(2*Sinh[h]^2)*( 
    Cosh[2*zSt]+bp*St^2*(-bp-Tanh[h]^2)*Cosh[2*zSt]/Tanh[h] + 2*bp*Sinh[2*zSt]
);*)
xdotsimplifiedforcompwithdiben = eps^2/(2*Sinh[h]^2)*Cosh[2*zSt]*( 
    1-(bp*St^2*((bp+Tanh[h]^2)-2*bp*Tanh[h]*Tanh[2*zSt]))/(Tanh[h]^2)
);
zdotsimplified = -St*bp/Tanh[h]*(
    1+eps^2/(2*Sinh[h]^2)*(Cosh[2*zSt]+Tanh[h]*Sinh[2*zSt])
);

(*Print[Collect[Expand[(xdotsimplified - (xdotavgdfinished /. theta -> Tanh[h]))], {St, eps}, Simplify]];*)
Print[Collect[Expand[(xdotsimplifiedforcompwithdiben - (xdotavgdfinished /. theta -> Tanh[h]))], {St, eps}, Simplify]];
Print[Collect[Expand[(zdotsimplified - (zdotavgdfinished /. theta -> Tanh[h]))], {St, eps}, Simplify]];

0
0
